In [1]:
from pyspark.sql import SparkSession 

from pyspark.sql.functions import *

from pyspark.sql.types import*

spark=SparkSession.builder.appName("Spark SQL Example").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/07 07:09:28 WARN Utils: Your hostname, Abhisheks-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.222.244.133 instead (on interface en0)
26/05/07 07:09:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 07:09:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Q1
Find employees earning more than their manager.

emp_id	name	salary	manager_id	dept_id
1	Alice	90000	3	D1
2	Bob	75000	3	D1
3	Carol	80000	NULL	D1
4	Dave	120000	5	D2
5	Eve	100000	NULL	D2
6	Frank	55000	3	D1

In [3]:
data = [
        (1, "Alice", 90000, 3,'D1'),
       (2, "Bob", 75000, 3,'D1'),
        (3, "Carol", 80000, None,'D1'),
        (4,"Dave",120000,5,'D2'),
        (5,"Eve",100000,None,'D2'),
        (6,"Frank",55000,3,'D1')
    
]

columns = ["emp_id", "name", "salary", "manager_id","dept_id"]


# create DataFrame

df = spark.createDataFrame(data, columns)
df.show()


+------+-----+------+----------+-------+
|emp_id| name|salary|manager_id|dept_id|
+------+-----+------+----------+-------+
|     1|Alice| 90000|         3|     D1|
|     2|  Bob| 75000|         3|     D1|
|     3|Carol| 80000|      NULL|     D1|
|     4| Dave|120000|         5|     D2|
|     5|  Eve|100000|      NULL|     D2|
|     6|Frank| 55000|         3|     D1|
+------+-----+------+----------+-------+



Alice manager is Carol and his salary is 80k and alice salary is 90k
Dave manager is eve and whoose salary is 100k and dave salary is 120k



our result will be 

Alice 
Carol 





In [4]:
df.createOrReplaceTempView("employees")



In [17]:
spark.sql("""
              select distinct e.name
              from employees e
              join employees m
              on e.manager_id = m.emp_id
              where e.salary >m.salary
          """).show()

+-----+
| name|
+-----+
| Dave|
|Alice|
+-----+



####  how self join works 

 step 1: Identify relation what is asked in question 

     emp ---->manger 

     employee references manager

    '''Relationship rule''': Whenever one column stores another row’s ID:

    
    Employee -----> Manager

    Because question talks about:  employee compared with manager

    

    




   step2:   match foriegn key to primary key 

           in emp table fk will e.manager_id and m.emp_id 

                       reference → actual row

              

🔥 MASTER RULE

Join condition usually remains SAME:  e.manager_id = m.emp_id


Because relationship never changes:  employee stores manager_id



In [18]:
# lets try with dataframe api

df.show()



+------+-----+------+----------+-------+
|emp_id| name|salary|manager_id|dept_id|
+------+-----+------+----------+-------+
|     1|Alice| 90000|         3|     D1|
|     2|  Bob| 75000|         3|     D1|
|     3|Carol| 80000|      NULL|     D1|
|     4| Dave|120000|         5|     D2|
|     5|  Eve|100000|      NULL|     D2|
|     6|Frank| 55000|         3|     D1|
+------+-----+------+----------+-------+



In [24]:
# what we need to  we need to perform selef join then filter condition 


# as we are doing self join we need two differnet table / data frames to join 


emp_df =df.alias("e")

manger_df=df.alias("m")

# without alias spark will not understant which one is emp and manager 


# now we will perfrom join 


result=emp_df.join(manger_df,col("e.manager_id")== col("m.emp_id"))\
             .filter(col("e.salary")>col("m.salary"))



In [ ]:
result.show()

+------+-----+------+----------+-------+------+-----+------+----------+-------+
|emp_id| name|salary|manager_id|dept_id|emp_id| name|salary|manager_id|dept_id|
+------+-----+------+----------+-------+------+-----+------+----------+-------+
|     1|Alice| 90000|         3|     D1|     3|Carol| 80000|      NULL|     D1|
|     4| Dave|120000|         5|     D2|     5|  Eve|100000|      NULL|     D2|
+------+-----+------+----------+-------+------+-----+------+----------+-------+



In [26]:
result.select(
       col("e.emp_id").alias("emplyooe_id"),
       col("e.name").alias("emp_name")
).show()

+-----------+--------+
|emplyooe_id|emp_name|
+-----------+--------+
|          1|   Alice|
|          4|    Dave|
+-----------+--------+



## customer who placed no order 

customer table :

| cust_id | name  | city   |
| ------- | ----- | ------ |
| 1       | Priya | Delhi  |
| 2       | Ravi  | Mumbai |
| 3       | Sneha | Pune   |
| 4       | Arjun | Delhi  |

orders table :

| order_id | cust_id | amount |
| -------- | ------- | ------ |
| 101      | 1       | 500    |
| 102      | 1       | 300    |
| 103      | 2       | 1200   |
| 104      | 3       | 800    |

find customer who has not place any order 

so i need to select those cust_id which not in orders table 




In [2]:
# lets create dataframe first 

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Customer Data
customer_data = [
    (1, "Priya", "Delhi"),
    (2, "Ravi", "Mumbai"),
    (3, "Sneha", "Pune"),
    (4, "Arjun", "Delhi")
]

customer_columns = ["cust_id", "name", "city"]

customer_df = spark.createDataFrame(customer_data, customer_columns)

# Orders Data
orders_data = [
    (101, 1, 500),
    (102, 1, 300),
    (103, 2, 1200),
    (104, 3, 800)
]

orders_columns = ["order_id", "cust_id", "amount"]

orders_df = spark.createDataFrame(orders_data, orders_columns)

orders_df.show()

customer_df.show()

+--------+-------+------+
|order_id|cust_id|amount|
+--------+-------+------+
|     101|      1|   500|
|     102|      1|   300|
|     103|      2|  1200|
|     104|      3|   800|
+--------+-------+------+

+-------+-----+------+
|cust_id| name|  city|
+-------+-----+------+
|      1|Priya| Delhi|
|      2| Ravi|Mumbai|
|      3|Sneha|  Pune|
|      4|Arjun| Delhi|
+-------+-----+------+



In [9]:
# lets creat temp vieve on both table 

customer_df.createOrReplaceTempView("customer")

orders_df.createOrReplaceTempView("orders")

In [ ]:
spark.sql("""
    select name
    from customer
    where cust_id not in (
        select cust_id from orders
    )
""").show()

# query is correct but we can use here left jon and filter using null 

# WHERE cust_id NOT IN (1,2,NULL)

# SQL gets confused because comparison with NULL is unknown.

# result can be empty 

+-----+
| name|
+-----+
|Arjun|
+-----+



LEFT JOIN keeps all left-table rows. When no match exists, right-side columns become NULL. Filtering WHERE right.pk IS NULL isolates the unmatched rows. This is faster than NOT IN on large tables with NULLs.

In [24]:
# lets see using left join 

spark.sql("""
              select c.name
              from customer c
              left join orders o
              on c.cust_id=o.cust_id
              where o.cust_id is null
          """).show()

+-----+
| name|
+-----+
|Arjun|
+-----+



In [30]:
# lest do using data frame api 

result_df=customer_df.join(orders_df,
                   # condition
                    customer_df.cust_id==orders_df.cust_id,
                    # type of join 
                    'left'
                 ).filter(orders_df.cust_id.isNull()).select(customer_df.name)


In [32]:
result_df.show()

+-----+
| name|
+-----+
|Arjun|
+-----+



26/05/07 01:10:24 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3837287 ms exceeds timeout 120000 ms
26/05/07 01:10:24 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/07 01:10:25 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

Q3 / 25
Employee–manager pairs with department name


emp_id	 name	 manager_id	   dept_id
1	    Alice	   3	        10
2	    Bob	       3	        10
3	    Carol      NULL	        10
4	    Dave	   5	        20
5	    Eve	       NULL	        20

departments:
dept_id	  dept_name
10	      Engineering
20	      Sales

what i need 

empName  managerName  department_name 
Alice    Carol         Engineering



In [3]:
emp_data = [
       (1,"Alice",3,10),
       (2,"Bob",3,20),
       (3,"Carol",None,10),
       (4,"Dave",5,20),
       (5,"Eve",None,20)
      
    
]

emp_columns = ["emp_id", "name","manager_id","dept_id"]


dept_data=[
        (10,"Engineering"),
        (20,"Sales")
]
dept_columns=["dept_id","dept_name"]

emp_df = spark.createDataFrame(emp_data, emp_columns)
dept_df = spark.createDataFrame(dept_data, dept_columns)



In [6]:
# I need to find 
# empName  managerName  department_name 
# Alice    Carol         Engineering

# for all emp i need thier managerName irrespect to whither thier manager is null 

emp_df.createOrReplaceTempView("employee")

dept_df.createOrReplaceTempView("dept")



In [9]:
spark.sql("""
             select e.name,m.name,d.dept_name
             from employee e
             LEFT JOIN employee m ON e.manager_id=m.emp_id
             join dept d on e.dept_id =d.dept_id
            
             
            
          """).show()

+-----+-----+-----------+
| name| name|  dept_name|
+-----+-----+-----------+
|Carol| NULL|Engineering|
|Alice|Carol|Engineering|
|  Eve| NULL|      Sales|
| Dave|  Eve|      Sales|
|  Bob|Carol|      Sales|
+-----+-----+-----------+



Key insight
Use LEFT JOIN for the self-join so top-level managers (NULL manager_id) still appear. Use INNER JOIN for departments since every employee must belong to a department.

Products never ordered:

products:

prod_id	    name	    price
1	       Laptop	    50000
2	       Mouse	    500
3	       Keyboard	    1500
4	       Monitor	    15000
5	       Webcam	    2000

order_items:

order_id	prod_id	qty
1	         1	     2
1	         3	     1
2	         2	     5
3	         1	     1
4	         3	     2

In [10]:
# if i perform left join then i have col where order_item.prod_id will be null 
# and we need to return that product 


# Products data
products_data = [
    (1, "Laptop", 50000),
    (2, "Mouse", 500),
    (3, "Keyboard", 1500),
    (4, "Monitor", 15000),
    (5, "Webcam", 2000)
]

products_columns = ["prod_id", "name", "price"]

# Order items data
order_items_data = [
    (1, 1, 2),
    (1, 3, 1),
    (2, 2, 5),
    (3, 1, 1),
    (4, 3, 2)
]

order_items_columns = ["order_id", "prod_id", "qty"]

# now create data frame

product_df=spark.createDataFrame(products_data,products_columns)

order_item_df=spark.createDataFrame(order_items_data,order_items_columns)


In [11]:
#create tempview 

product_df.createOrReplaceTempView("product")

order_item_df.createTempView("order")



In [13]:
spark.sql("""
             select p.prod_id,p.name
             from product p
             left join order o
             on p.prod_id=o.prod_id
             
             where o.prod_id is null
             
          
          """).show()

+-------+-------+
|prod_id|   name|
+-------+-------+
|      4|Monitor|
|      5| Webcam|
+-------+-------+



#### NOTES

When ever we need to do self join :
 see realtionship which asked and do fk--pk on condition

And whenver asked for not in -- USE NOT EXISTS (SELECT 1)

or in same situation we can use left join and filter base on is null condition 